# 03 · Optical polarization Möbius strips
**TightFocusingSolutions | Vectorial diffraction and three-dimensional polarization topology**

Tutorial 02 displays local polarization ellipses on a Cartesian grid. Here we follow their principal axes along a closed curve, construct actual ruled ribbons, and test whether the axes return with the same or opposite sign. No plotting threshold is applied to the numerical topology calculation.

This standalone CPU notebook reproduces the five-integral convention of [Tutorial 01](https://github.com/mffg1993/TightFocusingSolutions/blob/main/tutorials/01_Thesis_Bessel_Integral_Decomposition.ipynb) and [Tutorial 02](https://github.com/mffg1993/TightFocusingSolutions/blob/main/tutorials/02_Focal_Plane_Polarization_Atlas_PyVista.ipynb). Run all cells in Colab; no repository clone or GPU is required. Interactive Plotly figures can be rotated and exported as HTML.

## 1. Physical context
Bauer et al. experimentally observed optical polarization Möbius strips in tightly focused q-plate-generated beams ([Science 347, 964–966 (2015)](https://doi.org/10.1126/science.1260635)). Three-dimensional circular-polarization singularities and related structures can also arise from focused Gaussian beams ([PRL 117, 013601 (2016)](https://doi.org/10.1103/PhysRevLett.117.013601)). The present atlas inputs are not assumed to reproduce those experiments exactly.

For a complex electric field $\mathbf E$, with time convention $e^{-i\omega t}$, define
$$I=\mathbf E^*\cdot\mathbf E,\qquad Q=\mathbf E\cdot\mathbf E.$$
At nonzero intensity, $Q=0$ identifies circular polarization in three dimensions (a C-point in a selected plane). This differs from a C-point of the transverse projection. A C-point may move off axis during focusing.

Let $\mathbf u=\mathbf E/\sqrt I$ and $\chi=\arg Q/2$. The principal semiaxes are
$$\boldsymbol\alpha=\Re(\mathbf u e^{-i\chi}),\qquad \boldsymbol\beta=\Im(\mathbf u e^{-i\chi}),\qquad \boldsymbol\gamma=\Im(\mathbf u^*\times\mathbf u).$$
Their simultaneous sign is arbitrary. They are orthogonal and $|\alpha|^2+|\beta|^2=1$. At a circular state the principal axes are undefined; at a linear state the minor axis vanishes.

### Closed contours and ribbons
For a closed curve $\mathbf r(t)$, unwrap the phase of $Q$ continuously before halving it. The integer winding $w=\Delta\arg Q/(2\pi)$ implies $\boldsymbol\alpha(2\pi)=(-1)^w\boldsymbol\alpha(0)$ (and similarly for the minor axis if nonzero). Odd winding gives a nonorientable director ribbon; even winding gives an orientable ribbon. This parity is **not** the geometric count of half-twists.

Construct a ruled surface
$$\mathbf S(t,v)=\mathbf r(t)+v a\,\hat{\boldsymbol\alpha}(t),\quad -1\le v\le1.$$
Here $a$ is a visualization half-width. An odd winding joins opposite edges at the seam. The surface may have self-intersections or degeneracies: parity alone does not certify an embedded Möbius surface. Avoid contours passing through field zeros or C-points; minor ribbons also require nonzero minor axes. All lengths below are in vacuum wavelengths.

**Dependencies:** NumPy, SciPy, Matplotlib, Plotly and ipywidgets (normally preinstalled in Colab). In a local Jupyter environment, install these packages before running the notebook.


In [ ]:
import numpy as np
from scipy.special import jv
from scipy.integrate import simpson
from scipy.optimize import root
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from functools import lru_cache

NA, n_medium, lambda0, f = 0.85, 1.0, 1.0, 1.0
R = np.array([1,-1j])/np.sqrt(2)
L = np.array([1, 1j])/np.sqrt(2)

def fp(beta):
    return [(1, np.cos(beta)*R/np.sqrt(2)),
            (-1, -np.sin(beta)*R/np.sqrt(2)), (0,L/np.sqrt(2))]
CASES = {'Star':fp(0), 'Monstar':fp(np.pi/4), 'Lemon':fp(np.pi/2)}

## 2. Five-integral focusing solver
The incident pupil is a coherent sum of $q^{|\ell|}e^{-q^2}e^{i\ell\phi}(a,b)$, where $q=\sin\theta/\sin\alpha$ and $\alpha=\arcsin(\mathrm{NA}/n)$. The full-Poincaré cases use the same Jones vectors and mixing convention as Tutorial 02:
$$\mathbf E_{in}=\frac{1}{\sqrt2}[(\cos b\,U_{+1}-\sin b\,U_{-1})\mathbf e_R+U_0\mathbf e_L].$$
The ideal aplanatic Richards–Wolf integral reduces to five Bessel orders $\ell,\ell\pm2,\ell\pm1$. The common weight is $G=\sqrt{\cos\theta}\sin\theta q^{|\ell|}e^{-q^2}e^{ikz\cos\theta}$. The longitudinal integrals include an additional $\sin\theta$. The code preserves the complex field, omits the constant lens phase, and does not normalize incident powers. For each contour we evaluate the radial integrals directly at its sample radii, without Cartesian interpolation. Radial interpolation is used only for the background map.

In [ ]:
def radial_integrals(ell, rho, z=0., nt=801):
    if not 0 < NA < n_medium: raise ValueError('Require 0 < NA < n')
    if nt < 3 or nt % 2 == 0: raise ValueError('Use an odd nt >= 3')
    th = np.linspace(0,np.arcsin(NA/n_medium),nt)
    st, ct = np.sin(th), np.cos(th)
    q = st/(NA/n_medium); k=2*np.pi*n_medium/lambda0
    g = np.sqrt(ct)*st*q**abs(ell)*np.exp(-q*q+1j*k*z*ct)
    out = np.empty((5,len(rho)),complex)
    # Chunk radial nodes to bound RAM on Colab.
    for start in range(0,len(rho),128):
        sl=slice(start,start+128); arg=k*rho[sl,None]*st
        for j,(order,weight) in enumerate(zip([ell,ell+2,ell-2,ell+1,ell-1],[1+ct,1-ct,1-ct,st,st])):
            out[j,sl]=simpson(jv(order,arg)*g*weight,x=th,axis=-1)
    out[:3]*=-np.pi*1j**(ell+1)*f*n_medium/lambda0
    out[3:]*=np.pi*1j**ell*f*n_medium/lambda0
    return out

def field(x,y,terms,z=0.,nt=801,nrad=None):
    x,y=np.broadcast_arrays(x,y); shape=x.shape
    r=np.hypot(x,y).ravel(); ph=np.arctan2(y,x).ravel()
    E=np.zeros((3,r.size),complex)
    if nrad is None:
        # Rounding only removes floating-point duplicates on centered circles.
        nodes,idx=np.unique(np.round(r,14),return_inverse=True)
    else: nodes=np.linspace(0,max(r.max(),1e-14),nrad)
    for ell,jones in terms:
        a,b=jones
        if abs(a)+abs(b)<1e-15: continue
        raw=radial_integrals(ell,nodes,z,nt)
        vals=raw[:,idx] if nrad is None else np.array([np.interp(r,nodes,v) for v in raw])
        I1,I2,I3,I4,I5=vals*np.exp(1j*np.array([ell,ell+2,ell-2,ell+1,ell-1])[:,None]*ph)
        E+=a*np.array([(2*I1+I2+I3)/2,(I2-I3)/(2j),I5-I4])
        E+=b*np.array([(I2-I3)/(2j),(2*I1-I2-I3)/2,1j*(I4+I5)])
    return E.reshape((3,)+shape)

def iq(E):
    return np.sum(abs(E)**2,axis=0),np.sum(E*E,axis=0)

## 3. Locate C-points and inspect the field
Map $|Q|/I$: zero denotes circular polarization, one denotes linear polarization. Candidate zeros are found from grid cells where both real and imaginary parts of $Q$ cross zero, then refined with a direct field evaluation. This is a candidate search, not a proof of completeness (closely spaced or tangential roots can be missed).

In [ ]:
def find_cpoints(terms, half=0.55, grid=81, z=0., nt=401):
    from scipy.optimize import least_squares
    x=np.linspace(-half,half,grid); X,Y=np.meshgrid(x,x)
    E=field(X,Y,terms,z,nt,nrad=1001); I,Q=iq(E)
    candidates=[]
    for j in range(grid-1):
        for i in range(grid-1):
            v=Q[j:j+2,i:i+2]
            if v.real.min()<=0<=v.real.max() and v.imag.min()<=0<=v.imag.max():
                candidates.append([(x[i]+x[i+1])/2,(x[j]+x[j+1])/2])
    roots=[]
    def fun(p):
        _,q=iq(field(p[0],p[1],terms,z,nt)); return [q.real,q.imag]
    for p in candidates:
        if any(np.linalg.norm(np.asarray(p)-r)<0.25*(x[1]-x[0]) for r in roots): continue
        # Restrict refinement to the candidate cell and its immediate neighborhood.
        # This avoids unphysical root excursions and very large Bessel arguments.
        delta=2*(x[1]-x[0]); p=np.asarray(p)
        lo=np.maximum(p-delta,-half); hi=np.minimum(p+delta,half)
        scale=max(float(I.max()),1e-30)
        sol=least_squares(lambda v: np.asarray(fun(v))/scale,p,bounds=(lo,hi),
                          ftol=1e-12,xtol=1e-12,gtol=1e-12,max_nfev=30)
        v=sol.x
        ii,qq=iq(field(*v,terms,z,nt))
        if np.max(abs(v))<=half and ii>1e-8*I.max() and abs(qq)/ii<1e-8 and all(np.linalg.norm(v-r)>1e-5 for r in roots): roots.append(v)
    return x,I,Q,np.array(roots).reshape(-1,2)

beam='Star' # Star, Monstar, Lemon
terms=CASES[beam]
x,I,Q,roots=find_cpoints(terms)
fig,axs=plt.subplots(1,3,figsize=(12,3.6),layout='constrained')
for ax,data,title,cmap in zip(axs,[I/I.max(),abs(Q)/np.maximum(I,1e-30),np.angle(Q)],['Normalized intensity','Circularity diagnostic |Q|/I','Phase of Q'],'viridis viridis twilight'.split()):
    im=ax.imshow(data,origin='lower',extent=[x[0],x[-1],x[0],x[-1]],cmap=cmap)
    if len(roots): ax.scatter(*roots.T,facecolors='none',edgecolors='white',s=45)
    ax.set(xlabel='x / λ₀',ylabel='y / λ₀',title=title); fig.colorbar(im,ax=ax)
plt.show(); print('Refined C-point candidates (x,y):',roots)

## 4. Contour topology
A circle is sampled including its repeated endpoint. We check that intensity and $|Q|/I$ stay above numerical tolerances, that adjacent phase steps are small, and that the resulting director closes with the predicted sign. These tests are necessary but not sufficient; refine contour sampling and quadrature, especially near singularities.

In [ ]:
def contour(terms, radius=.4, center=(0.,0.), z=0., samples=361, nt=801):
    if radius<=0 or samples<9: raise ValueError('Invalid contour')
    t=np.linspace(0,2*np.pi,samples)
    p=np.array([center[0]+radius*np.cos(t),center[1]+radius*np.sin(t),np.full_like(t,z)]).T
    p[-1]=p[0]
    E=field(p[:,0],p[:,1],terms,z,nt); I,Q=iq(E)
    if I.max()==0 or I.min()/I.max()<1e-10: raise ValueError('Contour meets a dark point')
    if np.min(abs(Q)/I)<1e-6: raise ValueError('Contour too close to circular polarization')
    phase=np.unwrap(np.angle(Q)); steps=np.diff(phase)
    if np.max(abs(steps))>np.pi/2: raise ValueError('Refine contour: phase step too large')
    w=int(np.rint((phase[-1]-phase[0])/(2*np.pi)))
    u=E/np.sqrt(I); rect=u*np.exp(-.5j*phase)
    a,b=rect.real.T,rect.imag.T
    assert np.allclose(a[-1],(-1)**w*a[0],atol=1e-8)
    return dict(p=p,E=E,I=I,Q=Q,phase=phase,a=a,b=b,w=w,minq=float(np.min(abs(Q)/I)),minI=float(I.min()/I.max()))

for name,terms in CASES.items():
    c=contour(terms)
    print(f'{name:8s} w={c["w"]:+d}, {"nonorientable" if c["w"]%2 else "orientable"}, min |Q|/I={c["minq"]:.4g}')

## 5. Construct the ribbons
The mesh is built with the final row identified with the first row, reversing the width index for odd winding. This explicitly creates the correct seam connectivity rather than simply drawing two separate edges. Choose constant-width directors or normalized physical semiaxis lengths. The scale is graphical, not a displacement of the optical field. For even winding there are two boundary loops; for odd winding there is one. A minor-axis ribbon is rejected if its direction is numerically undefined.

In [ ]:
def ribbon_mesh(c, axis='a', width=.10, constant=True, across=9):
    d=c[axis]; lengths=np.linalg.norm(d,axis=1)
    if lengths.min()<1e-6: raise ValueError('Axis undefined on contour; choose a different radius')
    if width<=0 or across<3: raise ValueError('Invalid ribbon width or resolution')
    if constant: d=d/lengths[:,None]
    p=c['p'][:-1]; d=d[:-1]; m=len(p)
    v=np.linspace(-1,1,across)
    pts=(p[:,None,:]+width*v[None,:,None]*d[:,None,:]).reshape(-1,3)
    faces=[]
    for i in range(m):
        j=(i+1)%m; flip=(j==0 and c['w']%2!=0)
        def nxt(k): return j*across+(across-1-k if flip else k)
        for k in range(across-1):
            a=i*across+k; b=a+1; cc=nxt(k); dd=nxt(k+1)
            faces.extend([(a,b,cc),(b,dd,cc)])
    return pts,np.array(faces)

def ribbon_figure(c, width=.10, constant=True, major=True, minor=False, ellipses=True, stride=18):
    fig=go.Figure(); p=c['p']
    fig.add_scatter3d(x=p[:,0],y=p[:,1],z=p[:,2],mode='lines',line=dict(color='#555555',width=4),name='Contour')
    for enabled,key,name,color in [(major,'a','Major axis ribbon','#20ae57'),(minor,'b','Minor axis ribbon','#ff941f')]:
        if not enabled: continue
        v,f=ribbon_mesh(c,key,width,constant)
        fig.add_mesh3d(x=v[:,0],y=v[:,1],z=v[:,2],i=f[:,0],j=f[:,1],k=f[:,2],color=color,opacity=.78,name=name,showlegend=True,lighting=dict(ambient=.75,diffuse=.65,specular=.15),flatshading=False)
        for sign in [-1,1]:
            d=c[key]; d=d/np.linalg.norm(d,axis=1)[:,None] if constant else d
            edge=p+sign*width*d
            fig.add_scatter3d(x=edge[:,0],y=edge[:,1],z=edge[:,2],mode='lines',line=dict(color=color,width=5),showlegend=False)
    if ellipses:
        t=np.linspace(0,2*np.pi,61)
        for j in range(0,len(p)-1,stride):
            e=c['E'][:,j]/np.sqrt(c['I'][j]); pts=p[j]+.065*(e.real[:,None]*np.cos(t)+e.imag[:,None]*np.sin(t)).T
            fig.add_scatter3d(x=pts[:,0],y=pts[:,1],z=pts[:,2],mode='lines',line=dict(color='#52627a',width=3),showlegend=False)
    fig.update_layout(title=f'Q winding {c["w"]:+d} · '+('Nonorientable' if c['w']%2 else 'Orientable'),scene=dict(aspectmode='data',xaxis_title='x / λ₀',yaxis_title='y / λ₀',zaxis_title='z / λ₀'),height=670,margin=dict(l=0,r=0,b=0,t=55))
    return fig

beam='Star'; radius=.4; center=(0.,0.); z=0.
c=contour(CASES[beam],radius,center,z)
fig=ribbon_figure(c,width=.13,major=True,minor=False,ellipses=True)
fig.show()

### 5.1 Compare all three incident textures
Modify the radius or center to explore the topology. The ribbon in each plot is constructed from the actual calculated field, not a prescribed mathematical Möbius strip.

In [ ]:
for name,terms in CASES.items():
    c=contour(terms,radius=.4)
    fig=ribbon_figure(c,width=.13,ellipses=False)
    fig.update_layout(title=f'{name} — winding {c["w"]:+d}, '+('nonorientable' if c['w']%2 else 'orientable'))
    fig.show()

## 6. Radius sweep and numerical verification
The charge inside an optical-axis-centered circle can change as it crosses C-points. Missing values indicate contours rejected by the numerical safety checks. Compare 181 and 361 samples and 401 and 801 polar-angle nodes at the chosen radius. These are convergence tests, not independent validation of the optical model.

In [ ]:
radii=np.linspace(.04,.9,35)
fig,ax=plt.subplots(figsize=(8,4))
for name,terms in CASES.items():
    winding=[]
    for r in radii:
        try: winding.append(contour(terms,r,samples=181,nt=401)['w'])
        except ValueError: winding.append(np.nan)
    ax.plot(radii,winding,'o-',ms=3,label=name)
    a=contour(terms,.4,samples=181,nt=401); b=contour(terms,.4,samples=361,nt=801)
    err=np.linalg.norm(a['E']-b['E'][:,::2])/np.linalg.norm(b['E'][:,::2])
    print(f'{name}: winding {a["w"]} → {b["w"]}; relative field error {err:.3e}')
ax.set(xlabel='Contour radius / λ₀',ylabel='Winding of Q'); ax.legend(); ax.grid(alpha=.2); plt.show()

## 7. Interactive controls and export
Run the following optional cell to obtain an interactive control panel in Colab/Jupyter. Select the incident beam, center, radius, axial position, ribbon half-width and axis visibility. Press **Render** to recalculate; then download the standalone HTML or the mesh as an OBJ file. An OBJ contains the explicit seam topology and can be inspected in Blender. It is an open surface, not a watertight 3D-printable solid.

In [ ]:
import ipywidgets as widgets
from pathlib import Path
from IPython.display import clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError: pass
sel=widgets.Dropdown(options=list(CASES),description='Beam')
def slider(value,lo,hi,step,desc): return widgets.FloatSlider(value=value,min=lo,max=hi,step=step,description=desc,continuous_update=False)
rad=slider(.4,.04,1.2,.02,'Radius'); cx=slider(0,-.8,.8,.02,'Center x'); cy=slider(0,-.8,.8,.02,'Center y'); zz=slider(0,-1,1,.05,'z'); ww=slider(.13,.02,.3,.01,'Half-width')
major=widgets.Checkbox(value=True,description='Major'); minor=widgets.Checkbox(value=False,description='Minor'); ell=widgets.Checkbox(value=True,description='Ellipses'); const=widgets.Checkbox(value=True,description='Constant width')
button=widgets.Button(description='Render'); export=widgets.Button(description='Export HTML + OBJ'); out=widgets.Output(); state={}
def render(_=None):
    with out:
        clear_output(wait=True); state.clear()
        try:
            c=contour(CASES[sel.value],rad.value,(cx.value,cy.value),zz.value)
            fig=ribbon_figure(c,ww.value,const.value,major.value,minor.value,ell.value)
            state.update(c=c,fig=fig,width=ww.value,constant=const.value,axes=[k for k,v in [('a',major.value),('b',minor.value)] if v])
            print('Winding:',c['w'],' | minimum |Q|/I:',c['minq']); fig.show()
        except ValueError as e: print(e)
def save(_):
    with out:
        if not state: print('Render a valid contour first'); return
        html=Path('polarization_ribbon.html'); state['fig'].write_html(html,include_plotlyjs=True)
        paths=[html]
        for key in state['axes']:
            v,f=ribbon_mesh(state['c'],key,state['width'],state['constant']); path=Path(f'ribbon_{key}.obj')
            with path.open('w') as h:
                for p in v: h.write('v %.12g %.12g %.12g\n'%tuple(p))
                for face in f+1: h.write('f %d %d %d\n'%tuple(face))
            paths.append(path)
        import zipfile
        with zipfile.ZipFile('polarization_ribbon_export.zip','w',zipfile.ZIP_DEFLATED) as zf:
            for p in paths: zf.write(p,p.name)
        try:
            from google.colab import files
            files.download('polarization_ribbon_export.zip')
        except ImportError: print('Saved polarization_ribbon_export.zip')
button.on_click(render); export.on_click(save)
display(widgets.VBox([sel,rad,cx,cy,zz,ww,widgets.HBox([major,minor,ell,const]),widgets.HBox([button,export]),out]))

## 8. Interpretation and next steps
The parity test concerns the nonorientability of the director line bundle on the chosen curve. Counting signed geometric half-twists requires an additional framing convention and checks for tangencies of the director to the contour; it is not inferred from the magnitude of the Q winding. In particular, the labels star, monstar and lemon refer to incident pupil textures, not guaranteed classifications of the focused field. Adjust the center to a refined C-point and test nearby radii. For axial propagation, the solver already accepts z; tracing the full C-line and selecting its principal normal plane would be a further extension.